# Predicting Regional Retail Trade in Kazakhstan

This is the second version of this project. The first attempt used the classic Walmart sales dataset (US, 2010–2012) — and my instructor was right to push back on it: data from another country, more than a decade old, doesn't say much about how I think about a problem that's actually relevant to where I live and work. So for this version I went and built my own dataset, sourced directly from Kazakhstan's Bureau of National Statistics, using data that's current as of 2024–2025.

## The problem

Every month, the Bureau of National Statistics (stat.gov.kz) publishes a report called *"Socio-Economic Development of the Republic of Kazakhstan"*. For each of the country's 20 regions (including the cities of Astana, Almaty, and Shymkent), it breaks down dozens of indicators — retail trade volume, inflation, household income, unemployment, capital investment, and more. I pulled four of these reports (February 2024, June 2024, June 2025, and December 2025) and assembled them into one table: 20 regions × 4 time points = 80 observations.

**Goal:** predict a region's monthly retail trade volume (in million KZT) from five indicators — population, average income per capita, inflation, capital investment, and the unemployment rate.

This isn't a textbook dataset built for teaching purposes. These are real numbers that go into actual government budget planning. If a language model had generated synthetic numbers from scratch for an exercise like this, the relationships between variables (population vs. income vs. retail volume) usually wouldn't hang together the way real economic data does — here they do, because they're real.

## Where the data comes from

I transcribed the figures by hand from the official PDF bulletins:
- [Socio-Economic Development of Kazakhstan, February 2024](https://stat.gov.kz/upload/iblock/002/iymgccgoyj2adx0lgbbu1ovcapdity11/Ж-01-01-М%2002%202024%20(рус).pdf)
- [Socio-Economic Development of Kazakhstan, June 2024](https://stat.gov.kz/upload/iblock/b16/u1k0y8ikgvnv2y0vgxzc0gg6cx4ru9f5/Ж-01-М%2006%202024%20(рус).pdf)
- [Socio-Economic Development of Kazakhstan, June 2025](https://stat.gov.kz/upload/iblock/c98/gfrc35fthdnpbtp34vzz4i4xu1dc4cz4/Ж-01-М%2006%202025%20(рус).pdf)
- [Socio-Economic Development of Kazakhstan, December 2025](https://stat.gov.kz/upload/iblock/082/mwq0xdd7o5ukymteea8y9n1kyvhg12xn/Ж-01-М%2012%202025%20(рус).pdf)

Income and unemployment are published quarterly rather than monthly, so for each time point I used the closest available quarter — a standard practice when you're combining indicators that get reported on different schedules.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('../data/kazakhstan_regional_retail_trade.csv')
df['Region_label'] = df['Region'].str.replace('_', ' ')
df.head(10)

## What's in the table

| Column | Meaning |
|---|---|
| `Region` | one of Kazakhstan's 20 regions (including the three cities of republican significance) |
| `Period` | the month-year of the snapshot |
| `Retail_Trade_mln_KZT` | **target variable** — retail trade volume for that month, million KZT |
| `Population_thousand` | regional population at the start of the period, thousands |
| `Income_per_capita_KZT` | average nominal monthly income per capita, nearest quarter, KZT |
| `CPI_YoY_percent` | year-over-year consumer price inflation, % |
| `Investment_mln_KZT` | fixed capital investment for that month, million KZT |
| `Unemployment_rate_percent` | unemployment rate, nearest quarter, % |

In [ ]:
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nRegions: {df['Region'].nunique()}")
print(f"Periods: {sorted(df['Period'].unique())}")
df.describe().round(2)

## First look — and an immediate problem

The first histograms made it obvious this data needed some work before it went anywhere near a linear regression. Almaty is the country's financial capital with 2.3 million people; Ulytau is a sparsely populated region with about 220,000. The gap in retail trade volume between the two reaches roughly 160x. A linear model fit on raw numbers like that will mostly be reacting to two or three huge cities rather than picking up the actual underlying relationship across all 20 regions.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Feature distributions across 20 regions (4 periods, n=80)', fontsize=14, fontweight='bold')
cols = ['Retail_Trade_mln_KZT','CPI_YoY_percent','Population_thousand',
        'Investment_mln_KZT','Income_per_capita_KZT','Unemployment_rate_percent']
colors = ['#1565C0','#43A047','#FB8C00','#8E24AA','#E53935','#00897B']
for ax, col, c in zip(axes.flatten(), cols, colors):
    ax.hist(df[col], bins=20, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11)
plt.tight_layout()
plt.savefig('../images/1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

`Retail_Trade`, `Population`, and `Investment` are all heavily right-skewed — a long tail driven by a handful of large cities. This is a normal pattern in regional economics: city sizes within a country tend to roughly follow a power law (Zipf's law territory), not a normal distribution.

**The fix** is to log-transform the variables that scale multiplicatively rather than additively. This isn't a cosmetic trick — it's the more honest way to model the question. "What percentage does retail trade change by if population grows 1%" is a question that makes sense everywhere, from Ulytau to Almaty. "How many million KZT does retail trade change by if population grows by a thousand people" doesn't — a thousand extra residents means something completely different in Almaty than it does in Ulytau.

In [ ]:
df['log_retail'] = np.log(df['Retail_Trade_mln_KZT'])
df['log_population'] = np.log(df['Population_thousand'])
df['log_investment'] = np.log(df['Investment_mln_KZT'])
df['log_income'] = np.log(df['Income_per_capita_KZT'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Retail_Trade before and after log transform', fontsize=13, fontweight='bold')
axes[0].hist(df['Retail_Trade_mln_KZT'], bins=25, color='#1565C0', alpha=0.85)
axes[0].set_title('Raw values (mln KZT)')
axes[1].hist(df['log_retail'], bins=25, color='#43A047', alpha=0.85)
axes[1].set_title('After log()')
plt.tight_layout()
plt.savefig('../images/1b_log_transform.png', dpi=150, bbox_inches='tight')
plt.show()

After the transform the distribution is much closer to normal, and the largest cities no longer dominate the picture on their own.

## Correlations

In [ ]:
corr_df = df[['log_retail','log_population','log_income','log_investment',
              'CPI_YoY_percent','Unemployment_rate_percent']].copy()
corr_df.columns = ['Retail(log)','Population(log)','Income(log)','Investment(log)','CPI_YoY','Unemployment']
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(8,6.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size':10,'fontweight':'bold'})
ax.set_title('Feature correlation (log scale)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

corr['Retail(log)'].sort_values(ascending=False)

The two strongest correlations with retail trade are investment (r≈0.67) and population (r≈0.65) — but they're high for different reasons, and I want to spell that out rather than just report the numbers.

**Population** has the most direct, almost mechanical link. Retail trade is the sum of individual purchases. More people physically making purchases means a higher combined total for the region. The causal direction here is about as unambiguous as economics gets.

**Capital investment** is less obvious, and more interesting. Investment isn't household spending — it's money flowing from business and government into new infrastructure: shopping centers, logistics, production facilities. A region attracting heavy investment is usually a region with a generally active, growing economy, and that growth tends to drag consumption along with it. The causality here probably runs both ways: investment builds the infrastructure that retail needs, and growing retail demand attracts further investment.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,5.5))
fig.suptitle('Population and income per capita vs retail trade (log-log)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.scatter(df['log_population'], df['log_retail'], alpha=0.6, s=40, color='#1565C0')
z = np.polyfit(df['log_population'], df['log_retail'], 1)
xline = np.linspace(df['log_population'].min(), df['log_population'].max(), 50)
ax.plot(xline, np.poly1d(z)(xline), 'r-', lw=2)
ax.set_xlabel('log(Population, thousand)')
ax.set_ylabel('log(Retail trade, mln KZT)')
ax.set_title(f"r = {corr.loc['Population(log)','Retail(log)']:.2f}")

ax = axes[1]
ax.scatter(df['log_income'], df['log_retail'], alpha=0.6, s=40, color='#E65100')
z = np.polyfit(df['log_income'], df['log_retail'], 1)
xline = np.linspace(df['log_income'].min(), df['log_income'].max(), 50)
ax.plot(xline, np.poly1d(z)(xline), 'r-', lw=2)
ax.set_xlabel('log(Income per capita, KZT/month)')
ax.set_ylabel('log(Retail trade, mln KZT)')
ax.set_title(f"r = {corr.loc['Income(log)','Retail(log)']:.2f}")

plt.tight_layout()
plt.savefig('../images/3_scatter_population_income.png', dpi=150, bbox_inches='tight')
plt.show()

**Income per capita** works through a different mechanism than population. It isn't about how many buyers there are — it's about how much spending power each one has, how much money is left over after covering basic needs. The mechanism here is close to the textbook consumption function from intro macroeconomics: higher disposable income means higher consumer spending. Worth noticing on the right-hand chart: Atyrau region, with its oil-driven income, sits above the trend line on income but not proportionally above it on retail trade — a lot of that money is likely going into savings or investment outside the region rather than into local stores.

## Who sells how much: regions in December 2025

In [ ]:
dec = df[df['Period']=='2025-12'].sort_values('Retail_Trade_mln_KZT', ascending=True)
fig, ax = plt.subplots(figsize=(9,9))
bars = ax.barh(dec['Region_label'], dec['Retail_Trade_mln_KZT']/1000, color='#1976D2', alpha=0.85)
bars[-1].set_color('#E53935')
bars[-2].set_color('#FB8C00')
ax.set_xlabel('Retail trade, billion KZT (December 2025)')
ax.set_title('Retail trade volume by region, December 2025', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/4_regions_dec2025.png', dpi=150, bbox_inches='tight')
plt.show()

Almaty (red) sells almost 2.3x more than Astana (orange) — even though the two cities' populations aren't that far apart (2.3 million vs. 1.6 million). Almaty has historically been the country's commercial and financial center, and that gap isn't statistical noise; it's a structural feature of Kazakhstan's economy that the model below will also pick up on.

## What's going on with inflation

In [ ]:
period_avg = df.groupby('Period')[['CPI_YoY_percent','Retail_Trade_mln_KZT']].mean()
fig, ax1 = plt.subplots(figsize=(9,5.5))
ax2 = ax1.twinx()
ax1.plot(period_avg.index, period_avg['CPI_YoY_percent'], 'o-', color='#C62828', linewidth=2, label='Inflation (CPI YoY, %)')
ax2.plot(period_avg.index, period_avg['Retail_Trade_mln_KZT']/1000, 's-', color='#1565C0', linewidth=2, label='Average retail trade (bln KZT)')
ax1.set_ylabel('CPI year-over-year, %', color='#C62828')
ax2.set_ylabel('Retail trade, billion KZT', color='#1565C0')
ax1.set_title('Inflation and average regional sales over time', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/5_cpi_trend.png', dpi=150, bbox_inches='tight')
plt.show()

It's worth not mixing up cause and effect here. Inflation in Kazakhstan stayed persistently high over this period (109–113% year-over-year), and nominal retail trade nearly doubled over the same two years. That doesn't mean people bought twice as much stuff — retail trade is measured in current prices, not inflation-adjusted ones. Part of that growth is genuine consumption growth; part of it is simply that a liter of milk costs more in 2025 than it did in 2024. The model doesn't separate the two, and I flag that honestly in the limitations section below.

## The model

Features: `log_population`, `log_income`, `CPI_YoY_percent`, `log_investment`, `Unemployment_rate_percent`.
Target: `log_retail`.

75/25 split, `random_state=42`.

In [ ]:
feature_cols = ['log_population','log_income','CPI_YoY_percent','log_investment','Unemployment_rate_percent']
target_col = 'log_retail'

X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")

model = LinearRegression()
model.fit(X_train, y_train)
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)
print("Fitted")

## Metrics

R² is the metric I'm using to compare train and test (MAE and RMSE aren't directly comparable to each other — they're scaled differently), but I'm also reporting them to give a sense of error magnitude in concrete terms.

In [ ]:
def show_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f}  (log units)")
    return mae, rmse, r2

mae_tr, rmse_tr, r2_tr = show_metrics(y_train, pred_train, "TRAIN")
mae_te, rmse_te, r2_te = show_metrics(y_test, pred_test, "TEST ")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
print(f"\n5-fold CV R²: mean={cv_scores.mean():.3f}, std={cv_scores.std():.3f}")
print(f"Per fold: {np.round(cv_scores,3)}")

pred_test_real = np.exp(pred_test)
y_test_real = np.exp(y_test)
mae_real = mean_absolute_error(y_test_real, pred_test_real)
mape = np.mean(np.abs((y_test_real - pred_test_real) / y_test_real)) * 100
print(f"\nIn real KZT: average error ≈ {mae_real:,.0f} mln KZT, MAPE ≈ {mape:.0f}%")

Test R² is 0.72: the model explains roughly 72% of the variance in log(retail trade) using five features — a reasonable result for 80 observations of fairly noisy regional economic data. The 5-fold CV scores bounce around quite a bit (0.05 to 0.88) — expected given the small n: the same region shows up across different periods, and when an unusual case (Atyrau, with its oil-driven income, is a good example) happens to land in the test fold, R² on that fold takes a hit.

A 40% MAPE sounds large in isolation, but for cross-regional comparisons spanning a 160x size range, it means the model is reasonably good at getting the *order of magnitude* right, not at predicting exact totals to the tenge — which is a fair ceiling for a linear model on 80 rows.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5.5))
fig.suptitle('Model performance (back-transformed to real KZT)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.scatter(y_test_real/1000, pred_test_real/1000, alpha=0.7, s=60, color='#1565C0')
lims = [0, max(y_test_real.max(), pred_test_real.max())/1000*1.1]
ax.plot(lims, lims, 'r--', lw=2)
ax.set_xlabel('Actual volume (bln KZT)')
ax.set_ylabel('Predicted volume (bln KZT)')
ax.set_title(f'Actual vs Predicted | R²={r2_te:.3f}')

residuals = y_test - pred_test
ax = axes[1]
ax.hist(residuals, bins=12, color='#EF5350', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=2, linestyle='--')
ax.set_xlabel('Residual (log units)')
ax.set_title('Residual distribution')

plt.tight_layout()
plt.savefig('../images/6_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## The coefficients — and what they actually mean

This is the part I think matters most. Because the model works in logs for four of the five features (`log_population`, `log_income`, `log_investment`, and the target `log_retail`), the coefficients on those are **elasticities** — "what percentage does retail trade change by when this feature changes by 1%." CPI and unemployment stayed in their original percentage-point units, so their coefficients read differently: as the absolute change in log(retail trade) per one additional percentage point.

In [ ]:
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': model.coef_}).sort_values('Coefficient', key=abs)
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9,5.5))
colors_bar = ['#1565C0' if c>0 else '#E53935' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar, alpha=0.9)
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Coefficient (elasticity for log-transformed features)')
ax.set_title('Contribution of each feature to the model', fontsize=13, fontweight='bold')
for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (0.02 if val>=0 else -0.02), bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val>=0 else 'right', fontsize=10)
plt.tight_layout()
plt.savefig('../images/7_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

Walking through each one, in plain terms rather than just "positive" or "negative":

**`log_income` ≈ 1.53** — the strongest effect in the model, and notably above 1, meaning a 1% increase in income is associated with retail trade growing by *more* than 1% (an income elasticity above one — economists would call the underlying spending pattern "luxury-good-like" at the regional level). This tracks with intuition: when people have more disposable income, they don't just buy proportionally more bread and milk — spending on optional things (eating out, electronics, clothing, entertainment) grows disproportionately faster than spending on necessities, and that pulls the overall elasticity above 1.

**`log_population` ≈ 1.22** — also above 1, which is less obvious than it sounds. If every resident simply bought the same fixed basket of goods, the elasticity would be exactly 1 (twice the people, twice the sales). An elasticity above 1 suggests that larger cities sell disproportionately more per capita than smaller ones — likely because Almaty and Astana aren't just retail markets for their own residents, they're also logistics and shopping hubs that pull in trade from neighboring, less populated regions.

**`Unemployment` ≈ −0.44** — exponentiating this (exp(−0.437) ≈ 0.65) means a one-percentage-point rise in unemployment is associated with about a 35% lower retail trade volume, holding everything else fixed. The mechanism is direct: no job means no paycheck, no paycheck means less to spend. This is the only feature with a negative sign in the model, which is exactly what basic economic reasoning would predict.

**`log_investment` ≈ 0.11** — the weakest of the meaningful effects, and also makes sense: fixed capital investment (factories, roads, infrastructure) tends to pay off in consumption with a lag of months or years, through new jobs and new infrastructure, not instantly. Within a single month, the direct effect is small, and the model reflects that honestly rather than overstating it.

**`CPI_YoY` ≈ 0.02** — positive but tiny. Exponentiated, each additional percentage point of inflation is associated with only about a 2.3% increase in nominal retail trade, holding the other variables fixed. That's consistent with what I noted in the inflation chart above: most of the headline inflation effect on nominal sales is already baked into the time trend between periods (2024 → 2025), so once the other variables are controlled for, CPI's standalone effect comes out small.

## Limitations — without sugarcoating it

I'm not going to pretend this model is production-ready for forecasting. Here's what's actually wrong with it:

**Small sample.** 80 observations sounds reasonable, but it's really 20 regions repeated 4 times — not 80 independent data points. Successive snapshots of the same region are strongly correlated with each other (a region that historically sells a lot will keep showing up that way in both February and December). The real statistical power here is closer to n=20 than n=80, and the CV spread (0.05 to 0.88 across folds) is a direct symptom of that.

**Nominal, not real, prices.** Retail trade is reported in current prices. The model doesn't separate inflation-driven growth from genuine increases in consumption volume. The cleaner fix would be to deflate retail trade by CPI before modeling — a solid next step if I extend this.

**Purely linear effects.** The model has no interaction terms — the effect of income on retail trade might genuinely differ between Almaty and a rural region like Turkestan (spending patterns in a metropolis and a largely agricultural region aren't the same), but a linear regression averages that into one global coefficient.

**No within-year seasonality control.** The four snapshots span different months (February, June, December) but there's no explicit seasonal feature — and December is traditionally a higher-spending month, which could be quietly distorting the CPI and investment coefficients.

**Unemployment barely varies.** Regional unemployment in Kazakhstan sits in a narrow 4.0–5.1% band, so there isn't much variation for the model to extract a strong signal from. The real effect might be larger than what's shown here — there's just not enough spread in the data to see it clearly.

## What I'd do next

If I were taking this further, I'd: (1) add more time points — ideally monthly data over 2–3 years instead of four snapshots, (2) switch to real (CPI-deflated) retail trade as the target, (3) try a model with regional dummy variables to capture structural differences between cities and rural oblasts, and (4) compare this linear model against a Random Forest to check whether the underlying relationships are actually as linear as I'm assuming here.